# Zambia GeoHub AI Platform — Demo Notebook

Run cells top-to-bottom to demo all three AI features:
1. **Dataset Summarizer** — plain-English summary of a live dataset
2. **AI Chatbot** — question answered from real data
3. **Report Generator** — Word + PDF saved to disk

**Prerequisites:**
```bash
pip install -r requirements.txt
cp .env.example .env  # then add your ANTHROPIC_API_KEY
```

In [ ]:
# Cell 1 - Setup: verify environment
import os, json
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv('ANTHROPIC_API_KEY', '')
assert api_key and not api_key.startswith('sk-ant-REPLACE'), 'ANTHROPIC_API_KEY not set. Edit your .env file.'
print('Environment OK')
print(f'   API key ends with: ...{api_key[-6:]}')

In [ ]:
# Cell 2 - Search GeoHub for datasets
from hub.client import HubClient
hub = HubClient()
results = hub.search_datasets('zambia health facilities', max_results=5)
print(f'Found {len(results)} datasets:\n')
for i, ds in enumerate(results, 1):
    print(f'{i}. {ds["name"]}')
    print(f'   {ds["description"][:120]}')
    print(f'   URL: {ds["url"]}')
    print()

In [ ]:
# Cell 3 - Fetch live GeoJSON
dataset = results[0]
print(f'Loading: {dataset["name"]}')
geojson = hub.fetch_geojson(dataset['url'], max_features=50)
print(f'Loaded {len(geojson["features"])} features')
if geojson['features'] and geojson['features'][0].get('geometry'):
    print(f'Geometry type: {geojson["features"][0]["geometry"]["type"]}')
print()
print('Sample properties (first feature):')
if geojson['features']:
    print(json.dumps(geojson['features'][0]['properties'], indent=2))

In [ ]:
# Cell 4 - FEATURE 3: Dataset Summarizer
from ai.claude_client import ClaudeClient
from ai.prompts import summarizer_system_prompt, summarizer_prompt
from utils.geo_utils import summarize_geojson, geojson_to_sample_rows

claude = ClaudeClient()
stats = summarize_geojson(geojson)
samples = geojson_to_sample_rows(geojson, n=5)

print(f'Stats: {stats["feature_count"]} features, {stats["geometry_type"]} geometry, {len(stats["fields"])} fields')

prompt = summarizer_prompt(
    dataset['name'], dataset['description'],
    dataset['fields'], samples, stats['feature_count']
)

print('\nGenerating AI summary...\n')
summary = claude.ask(system=summarizer_system_prompt(), user=prompt, max_tokens=1024)

print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(summary)

In [ ]:
# Cell 5 - FEATURE 1: AI Chatbot
from ai.prompts import chatbot_system_prompt, chatbot_user_prompt

question = 'Which areas in Zambia have the highest concentration of health facilities?'

chat_datasets = hub.search_datasets(question, max_results=3)
chat_geojson = hub.fetch_geojson(chat_datasets[0]['url'], max_features=100)
chat_samples = geojson_to_sample_rows(chat_geojson, n=5)

print(f'Question: {question}')
print(f'\nDatasets found: {", ".join(ds["name"] for ds in chat_datasets)}')
print('\nGenerating answer...\n')

answer = claude.ask(
    system=chatbot_system_prompt(),
    user=chatbot_user_prompt(question, chat_datasets, chat_samples),
    max_tokens=1500
)

print('=' * 60)
print('AI ANSWER')
print('=' * 60)
print(answer)

In [ ]:
# Cell 6 - FEATURE 2: Report Generator — saves demo_report.docx + demo_report.pdf
from ai.prompts import report_system_prompt, report_prompt
from reports.builder import ReportBuilder

rpt_samples = geojson_to_sample_rows(geojson, n=10)
rpt_prompt = report_prompt(
    dataset['name'], dataset['description'],
    dataset['fields'], stats, rpt_samples
)

print('Generating report with Claude...')
report_text = claude.ask(system=report_system_prompt(), user=rpt_prompt, max_tokens=3000)

print('Building Word and PDF files...')
builder = ReportBuilder()
docx_bytes = builder.to_docx(title=dataset['name'], ai_content=report_text, dataset_meta=dataset)
pdf_bytes  = builder.to_pdf(title=dataset['name'], ai_content=report_text, dataset_meta=dataset)

with open('demo_report.docx', 'wb') as f:
    f.write(docx_bytes)
with open('demo_report.pdf', 'wb') as f:
    f.write(pdf_bytes)

print(f'\nReports saved:')
print(f'  demo_report.docx ({len(docx_bytes):,} bytes)')
print(f'  demo_report.pdf  ({len(pdf_bytes):,} bytes)')

In [ ]:
# Cell 7 - Map preview (renders inline in Jupyter)
from utils.geo_utils import make_folium_map
m = make_folium_map(geojson, dataset['name'])
print(f'Map for: {dataset["name"]}')
m

In [ ]:
# Cell 8 - Launch the Streamlit web app
print('To run the full Streamlit web app, open a terminal and run:')
print()
print('    streamlit run app.py')
print()
print('The app opens at: http://localhost:8501')
print()
print('Tabs:')
print('  Tab 1 - AI Chatbot        (streaming + maps)')
print('  Tab 2 - Report Generator  (Word + PDF downloads)')
print('  Tab 3 - Dataset Summarizer (metrics + field table + map)')